# Causal Mask

**In a Decoder-only model (like GPT), the Causal Mask is *always* active, even for the user prompt.**

Here is the breakdown of why that is and how it actually plays out during "Prefill" vs. "Generation."


### 1. The "Always-On" Mask

In a Decoder-only architecture, the model doesn't have a separate "Prompt Processor" and "Token Generator" circuit. It’s the same stack of Transformer blocks.

To keep the math consistent, we apply the **Causal Mask** to the entire sequence (Prompt + Generated Tokens).

* **Token 1** of your prompt can only see itself.
* **Token 2** can see Token 1 and itself.
* **Token ** of your prompt can see all previous tokens.

**Why?** Because the model was *trained* that way. If you suddenly "turned off" the mask for the prompt during inference, the mathematical distribution of the attention scores would change, and the model would likely output gibberish. It expects every token to only "know" what came before it.


### 2. Prefill vs. Autoregression

Even though the mask is always there, the **hardware** treats these two stages differently for efficiency:

#### Phase A: The Prefill (Processing the Prompt)

When you hit enter, the GPU takes your entire prompt (say, 50 words) and processes it in **one parallel pass**.

* Even though the Causal Mask is applied (preventing Word 2 from seeing Word 10), the GPU calculates the attention for all 50 words simultaneously.
* This is highly efficient. We calculate the  and  vectors for every word in the prompt and store them in a **KV Cache**.

#### Phase B: The Autoregressive Generation (One by One)

Now the model needs to generate the *next* token.

* It takes the last vector, looks at the **KV Cache** (the "memory" of the prompt), and predicts one new token.
* It then appends that new token to the sequence and repeats.
* In this phase, we aren't re-calculating the whole prompt—we just look at the new token's relationship to the previous ones.


### 3. The "Full Context" Illusion

You might ask: *"If the first word of my prompt can't see the rest of the prompt, isn't it losing information?"*

Technically, yes, the first word's representation is "weak." However, by the time the model gets to the **last word** of your prompt, that word has had the opportunity to attend to **every single word** you wrote. Since the model uses that final, fully-informed representation to predict the first generated token, the "knowledge" of the full prompt is preserved.


### Why the "Add & Norm"?

* **The "Add" (Residual Connection):** This allows gradients to flow through the network without disappearing. It’s like a "highway" that lets the original input skip the layer if the layer isn't helpful yet.
* **The "Norm" (Layer Normalization):** This keeps the values (activations) in a healthy range (roughly mean 0, variance 1), which prevents the model's "brain" from exploding during deep calculations.

Would you like to see how we stack these blocks and add **Positional Encodings** so the model knows the *order* of the words?

### How the Mask Works Mathematically
We modify the attention score calculation by adding a Mask Matrix ($M$) before the Softmax step:



$$\text{Attention}(Q, K, V) = \text{softmax} \left( \frac{QK^T}{\sqrt{d_k}} + M \right) V$$

The Mask $M$ is a square matrix of the same size as our attention scores ($L \times L$). It is filled with:

* **0** for positions we want to keep (the past and present).
* $-\infty$ for positions we want to hide (the future).

When we apply softmax, $e^{-\infty}$ becomes $0$, effectively canceling out any contribution from future words.



### Putting it into a Transformer Block

To see how this fits into the bigger picture, here is the "Wrapper" that turns our Multi-Head Attention into a functional **Transformer Block**. This includes **Layer Normalization** and the **Feed-Forward Network (FFN)**.


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class TransformerBlock(nn.Module):
    def __init__(self, embed_size, heads, dropout, forward_expansion):
        super(TransformerBlock, self).__init__()
        self.attention = MultiHeadAttention(embed_size, heads)
        self.norm1 = nn.LayerNorm(embed_size)
        self.norm2 = nn.LayerNorm(embed_size)

        # The Feed-Forward part: Two linear layers with a ReLU in between
        self.feed_forward = nn.Sequential(
            nn.Linear(embed_size, forward_expansion * embed_size),
            nn.ReLU(),
            nn.Linear(forward_expansion * embed_size, embed_size),
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, value, key, query, mask):
        # 1. Self-Attention + Residual Connection
        attention = self.attention(value, key, query, mask)
        x = self.norm1(attention + query) # Add & Norm
        
        # 2. Feed-Forward + Residual Connection
        forward = self.feed_forward(x)
        out = self.norm2(forward + x) # Add & Norm
        
        return out


/Users/jaero/projects/interviews/scale_ai/.venv/lib/python3.14/site-packages/torch/_subclasses/functional_tensor.py:283: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))
